Latent Dirichlet

Named Entity Recognition

In [1]:
import random
import spacy
from spacy import displacy
from spacy.training import Example
from spacy.training.iob_utils import offsets_to_biluo_tags
import jsonlines
import logging
from spacy.scorer import Scorer

In [3]:

nlp = spacy.load("en_core_web_sm")
nlp2 = spacy.load("fr_core_news_sm")

matcher = Matcher(nlp.vocab)
octet_rx = r'(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)'
pattern= [ {"TEXT": {"REGEX": r"^{0}(?:\.{0}){{3}}$".format(octet_rx)}}]
matcher.add("ip",[pattern])

doc = nlp("Link down , Bypass (92.33.222.88)  is not pinging")
matches = matcher(doc)
for match_id, start, end in matches:
    string_id = nlp.vocab.strings[match_id]
    span = doc[start:end]
    print(match_id, string_id, start, end, span.text)


1699727618213446713 ip 5 6 92.33.222.88


In [7]:
#import spacy
import datefinder
import ipaddress
import re

def extract_dates_hours_ips(text):
    
    common_os = [
        "debian",
        "ubuntu",
        "windows",
        "macos",
        "ios",
        "android",
        "centos",
        "fedora",
        "redhat",
        "suse",
        "arch",
        "mint",]
    
    # Expressions régulières pour extraire les dates et heures
    date_pattern = r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"  # Format JJ/MM/AAAA ou JJ-MM-AAAA
    time_pattern = r"\b\d{1,2}h\d{0,2}\b"  # Format HHhMM
    
    # Utiliser datefinder pour extraire les dates et heures
    matches = datefinder.find_dates(text, source=True)
    dates = [match[0] for match in matches if match[1] == 'date']
    hours = [match[0] for match in matches if match[1] == 'time']

    # Charger le modèle spaCy pour le français
    nlp = spacy.load("fr_core_news_sm")

    # Traiter le texte avec spaCy
    doc = nlp(text)

    # Initialiser les listes pour stocker les entités trouvées
    ips = []
    names = []
    operating_systems = []

    # Parcourir toutes les entités identifiées par spaCy
    for ent in doc.ents:
        if ent.label_ == "PER":
            if " " in ent.text:
                names.append(ent.text)
        # Vérifier si l'entité est un système d'exploitation    
        elif ent.label_ == "MISC":
            # Vérifier si c'est un système d'exploitation commun
            if ent.text.lower() in common_os:
                operating_systems.append(ent.text)
            # Ajouter tous les autres systèmes d'exploitation du label "MISC"
            else:
                operating_systems.append("others")

    # Utiliser une expression régulière pour extraire les adresses IP
    ip_pattern = r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b"
    ips.extend(re.findall(ip_pattern, text))
    
    # Utiliser les expressions régulières pour extraire les dates et heures
    dates.extend(re.findall(date_pattern, text))
    hours.extend(re.findall(time_pattern, text))
    
    return dates, hours, ips, names, operating_systems

# Texte à analyser
text = "Dans notre entreprise, tous les ordinateurs sont sous Debian. Notre développeur Jean-Christophe LAFLEUR est autorisé. L'entreprise est ouverte de 8h à 21h du lundi au vendredi. La plage des adresses IP est le 10.0.0.1/24."

# Appel de la fonction pour extraire les entités
dates, hours, ips, names, operating_systems = extract_dates_hours_ips(text)

# Afficher les résultats
print("Dates trouvées :", dates)
print("Heures trouvées :", hours)
print("Adresses IP trouvées :", ips)
print("Noms trouvés :", names)
print("Systèmes d'exploitation trouvés :", operating_systems)


Dates trouvées : []
Heures trouvées : ['8h', '21h']
Adresses IP trouvées : ['10.0.0.1']
Noms trouvés : ['Jean-Christophe LAFLEUR']
Systèmes d'exploitation trouvés : ['Debian']


In [8]:
import random
from spacy.training.example import Example

# Définir les exemples d'entraînement avec les annotations
TRAIN_DATA = [
    ("Notre entreprise utilise Windows comme système d'exploitation.", {"entities": [(27, 34, "OS")]}),
    ("Les développeurs utilisent Mac OS pour le développement.", {"entities": [(24, 30, "OS")]}),
    ("Linux est préféré pour les serveurs de l'entreprise.", {"entities": [(0, 5, "OS")]}),
    ("Windows 10 est la dernière version du système d'exploitation Windows.", {"entities": [(0, 12, "OS")]}),
]

def train_ner_model(nlp, train_data, n_iter=100):
    # Créer un objectif NER vide
    ner = nlp.get_pipe("ner")
    if not ner:
        ner = nlp.create_pipe("ner")
        nlp.add_pipe(ner, last=True)

    # Ajouter les annotations d'entraînement au modèle
    for _, annotations in train_data:
        for ent in annotations.get("entities"):
            ner.add_label(ent[2])

    # Entraîner le modèle NER avec les données d'entraînement
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.begin_training()
        for itn in range(n_iter):
            random.shuffle(train_data)
            losses = {}
            for text, annotations in train_data:
                example = Example.from_dict(nlp.make_doc(text), annotations)
                nlp.update([example], drop=0.5, sgd=optimizer, losses=losses)
            print(f"Iteration {itn+1}: Losses - {losses}")

    return nlp

# Charger le modèle spaCy pour le français
nlp = spacy.load("fr_core_news_lg")

# Entraîner le modèle NER personnalisé
nlp = train_ner_model(nlp, TRAIN_DATA)
print(nlp.pipe_labels)
# Texte à analyser
text = "Notre entreprise utilise Windows 10 comme système d'exploitation."

# Traiter le texte avec le modèle entraîné
doc = nlp(text)

# Extraire les entités nommées
for ent in doc.ents:
    print(ent.text, ent.label_)

/home/jc/.local/lib/python3.10/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Notre entreprise utilise Windows comme système d'e..." with entities "[(27, 34, 'OS')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/jc/.local/lib/python3.10/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Les développeurs utilisent Mac OS pour le développ..." with entities "[(24, 30, 'OS')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/jc/.local/lib/python3.10/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Windows 10 est la dernière version du 

Iteration 1: Losses - {'ner': 26.268152955919504}
Iteration 2: Losses - {'ner': 30.127052411437035}
Iteration 3: Losses - {'ner': 26.16173079609871}
Iteration 4: Losses - {'ner': 19.959613766521215}
Iteration 5: Losses - {'ner': 9.758254643529654}
Iteration 6: Losses - {'ner': 2.995469444154878}
Iteration 7: Losses - {'ner': 2.306081092913706}
Iteration 8: Losses - {'ner': 2.024720371058967}
Iteration 9: Losses - {'ner': 1.9806541149667831}
Iteration 10: Losses - {'ner': 1.998139588144075}
Iteration 11: Losses - {'ner': 1.8739646684074305}
Iteration 12: Losses - {'ner': 1.9992528879126619}
Iteration 13: Losses - {'ner': 2.002289036616074}
Iteration 14: Losses - {'ner': 1.9634736425766106}
Iteration 15: Losses - {'ner': 1.845947449082436}
Iteration 16: Losses - {'ner': 1.786575814208949}
Iteration 17: Losses - {'ner': 1.9924485647169403}
Iteration 18: Losses - {'ner': 1.676689384995428}
Iteration 19: Losses - {'ner': 1.768408441148124}
Iteration 20: Losses - {'ner': 1.747041826176807}
I

In [9]:
#text2 = "Windows 10 est la dernière version du système d'exploitation Windows."
doc2 = nlp2("Dans notre entreprise, tous les ordinateurs sont sous Debian. Notre développeur Jean-Christophe Lafleur est autorisé.  L'entreprise est ouvert de 8h à 21h du lundi au vendredi. La plage des adresses IP est le 10.0.0.1/24.")

# Traiter le texte avec le modèle entraîné
#doc2 = nlp(text2)

# Extraire les entités nommées
for ent in doc2.ents:
    print(ent.text, ent.label_)

Debian MISC
Jean-Christophe Lafleur PER


In [10]:
displacy.render(doc2, style="ent", jupyter=True)

In [2]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')


def evaluate_model(nlp, eval_data):
    scorer = Scorer()
    examples = []
    for text, annotations in eval_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        example.predicted = nlp(str(example.predicted))
        examples.append(example)
    return scorer.score(examples)

def load_training_data_from_jsonl(file_path):
        train_data = []
        with jsonlines.open(file_path) as reader:
            for item in reader:
                text = item['text']
                entities = [(entity['start_offset'], entity['end_offset'], entity['label']) for entity in item['entities']]
                annotations = {"entities": entities}
                train_data.append((text, annotations))
        return train_data

def print_doc_entities(_doc):
    if _doc.ents:
        for _ent in _doc.ents:
            print(f"     {_ent.text} {_ent.label_}")
        displacy.render(_doc, style="ent", jupyter=True)         
    else:
        print("     NONE")

def customizing_pipeline_component(nlp, train_data, iterations, eval_data=None):
    # Get the NER component from the pipeline
        
    if 'ner' not in nlp.pipe_names:
        nlp.add_pipe("ner", last=True)
    ner = nlp.get_pipe("ner")
    # Add the NER labels
    for _, annotations in train_data:
        for ent in annotations.get("entities"):
            ner.add_label(ent[2])
                
    print("   Training ...")
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.begin_training()
        #print(optimizer.learn_rate)
        for iteration in range(iterations):
            random.shuffle(train_data)
            losses = {}
            for text, annotations in train_data:
                example = Example.from_dict(nlp.make_doc(text), annotations)
                nlp.update([example], drop=0.5, sgd=optimizer, losses=losses)
            if eval_data:
                scores = evaluate_model(nlp, eval_data)
                logging.info(f"Iteration {iteration + 1}: Losses - {losses}, Scores - Precision: {scores['ents_p']}, Recall: {scores['ents_r']}, F1-score: {scores['ents_f']}")
            else:
                logging.info(f"Iteration {iteration + 1}: Losses - {losses}")
    logging.info("Custom pipeline successfully.")
    # Enable all previously disabled pipe components
    """for pipe_name in other_pipes:
        nlp.enable_pipe(pipe_name)"""
        
    return nlp

def save_model(nlp, model_output):   
    return nlp.to_disk(model_output)

In [10]:
nlp = spacy.load('fr_core_news_lg')
# Result before training
#print(f"\nResult BEFORE training:")
#doc = nlp(u'Windows 10 est la dernière version du système d\'exploitation Windows. Jean Dupont est un employé. Le pare-feu matériel FortiGate bloque les connexions vers les plages d\'adresses IP 10.0.0.0/8 et 192.168.0.0/16. Configurez le routeur Linksys avec l\'adresse IP 192.168.0.1 pour accéder à son interface d\'administration.Le DNS est responsable de traduire les noms de domaine en adresses IP, ce qui permet aux utilisateurs d\'accéder à des sites web tels que \'www.google.com\' en utilisant le lien \'172.217.167.174\'.')
#print_doc_entities(doc)


In [8]:
TRAIN_DATA = load_training_data_from_jsonl("admin.jsonl")
eval_data = load_training_data_from_jsonl('eval.jsonl')

In [11]:
nlp=customizing_pipeline_component(nlp, TRAIN_DATA, 100,eval_data)

[2023-07-30 16:28:15,175] [INFO] Created vocabulary
2023-07-30 16:28:15,175 - INFO - Created vocabulary
[2023-07-30 16:28:15,181] [INFO] Finished initializing nlp object
2023-07-30 16:28:15,181 - INFO - Finished initializing nlp object


   Training ...


2023-07-30 16:28:28,750 - INFO - Iteration 1: Losses - {'ner': 1046.5226454111034}, Scores - Precision: 0.11678832116788321, Recall: 0.09696969696969697, F1-score: 0.10596026490066227
2023-07-30 16:28:39,702 - INFO - Iteration 2: Losses - {'ner': 758.8869167108597}, Scores - Precision: 0.2595419847328244, Recall: 0.20606060606060606, F1-score: 0.22972972972972971
2023-07-30 16:28:50,722 - INFO - Iteration 3: Losses - {'ner': 770.8537017638146}, Scores - Precision: 0.31690140845070425, Recall: 0.2727272727272727, F1-score: 0.2931596091205212
2023-07-30 16:29:02,196 - INFO - Iteration 4: Losses - {'ner': 600.4312171200464}, Scores - Precision: 0.4268292682926829, Recall: 0.42424242424242425, F1-score: 0.425531914893617
2023-07-30 16:29:15,277 - INFO - Iteration 5: Losses - {'ner': 563.8355014749097}, Scores - Precision: 0.5337423312883436, Recall: 0.5272727272727272, F1-score: 0.5304878048780487
2023-07-30 16:29:27,290 - INFO - Iteration 6: Losses - {'ner': 439.3604048923121}, Scores - P

In [12]:
nlp2 = spacy.blank('fr')
nlp2=customizing_pipeline_component(nlp2, TRAIN_DATA, 100, eval_data)

[2023-07-30 16:49:26,397] [INFO] Created vocabulary
2023-07-30 16:49:26,397 - INFO - Created vocabulary
[2023-07-30 16:49:26,400] [INFO] Finished initializing nlp object
2023-07-30 16:49:26,400 - INFO - Finished initializing nlp object


   Training ...


2023-07-30 16:49:36,584 - INFO - Iteration 1: Losses - {'ner': 1102.406719453246}, Scores - Precision: 0.13559322033898305, Recall: 0.09696969696969697, F1-score: 0.11307420494699648
2023-07-30 16:49:45,594 - INFO - Iteration 2: Losses - {'ner': 744.8883876901037}, Scores - Precision: 0.3464566929133858, Recall: 0.26666666666666666, F1-score: 0.3013698630136986
2023-07-30 16:49:54,896 - INFO - Iteration 3: Losses - {'ner': 619.7727934673999}, Scores - Precision: 0.36879432624113473, Recall: 0.3151515151515151, F1-score: 0.33986928104575165
2023-07-30 16:50:04,409 - INFO - Iteration 4: Losses - {'ner': 598.4118980801113}, Scores - Precision: 0.5064935064935064, Recall: 0.4727272727272727, F1-score: 0.48902821316614414
2023-07-30 16:50:14,031 - INFO - Iteration 5: Losses - {'ner': 506.3996033378474}, Scores - Precision: 0.46060606060606063, Recall: 0.46060606060606063, F1-score: 0.46060606060606063
2023-07-30 16:50:23,759 - INFO - Iteration 6: Losses - {'ner': 478.845174395204}, Scores -

In [13]:
# Result after training
print(f"Result AFTER training:")
doc = nlp(u'Windows 10 est la dernière version du système d\'exploitation Windows.Le pare-feu matériel FortiGate bloque les connexions vers les plages d\'adresses IP 10.0.0.0/8 et 192.168.0.0/16. Dans l\'entreprise XYZ Inc., les serveurs tournent sous Windows Server 2019 pour assurer la stabilité de leur infrastructure réseau. La PME "Techno Innovations" fermera ses portes du 10 au 20 juillet pour participer à une importante conférence internationale sur les nouvelles technologies. Configurez le routeur Linksys avec l\'adresse IP 192.168.0.1 pour accéder à son interface d\'administration. Jean Dupont est un employé. J\'ai été impressionné par les performances améliorées de mon ordinateur depuis que je suis passé à Windows 11. Les imprimantes du sous-réseau 172.16.0.0/16 sont accessibles depuis n\'importe quel appareil du réseau local. Le DNS est responsable de traduire les noms de domaine en adresses IP, ce qui permet aux utilisateurs d\'accéder à des sites web tels que \'www.google.com\' en utilisant le lien \'172.217.167.174\'. Le pare-feu logiciel pfSense est largement utilisé dans les environnements réseau complexes avec la plage d\'adresses IP 172.16.0.0/20. Chez JKLM Hosting, les serveurs tournent sous NixOS 21.05 pour son approche déclarative et son système de gestion de paquets innovant. \"Saveurs d\'Asie\", une PME de restauration asiatique, est ouverte tous les jours de la semaine, de 11h30 à 22h00, pour régaler les amateurs de cuisine asiatique. Le pare-feu logiciel pfSense est largement utilisé dans les environnements réseau complexes avec la plage d\'adresses IP 172.16.0.0/20, et l\'entreprise utilise FreeBSD 13 pour ses serveurs, appréciant la flexibilité et les performances de ce système d\'exploitation open-source.')
print_doc_entities(doc)
"""    
colors = {'OS': "#85C1E9", "PER": "#ff6961", "firewall": "#75C2F6","IP_firewall": "#A076F9", "routeur": "#E9B384", "IP_routeur": "#FFECAF" }
options = {"ents": ['OS', 'PER','firewall','IP_firewall','routeur','IP_routeur'],"colors": colors}"""

Result AFTER training:
     Windows 10 OS
     Windows OS
     FortiGate Firewall
     10.0.0.0/8 IP_Firewall
     192.168.0.0/16 IP_Firewall
     Windows Server 2019 OS_serveur
     10 au 20 juillet Date
     Linksys Routeur
     192.168.0.1 IP_Routeur
     Jean Dupont Personne
     Windows 11 OS
     172.16.0.0/16 IP_Subnet
     www.google.com DNS
     172.217.167.174 IP_DNS
     pfSense Firewall
     172.16.0.0/20 IP_Firewall
     NixOS 21.05 OS_serveur
     semaine Days
     11h30 Time_Start
     22h00 Time_End
     pfSense Firewall
     172.16.0.0/20 IP_Firewall
     FreeBSD 13 OS_serveur


'    \ncolors = {\'OS\': "#85C1E9", "PER": "#ff6961", "firewall": "#75C2F6","IP_firewall": "#A076F9", "routeur": "#E9B384", "IP_routeur": "#FFECAF" }\noptions = {"ents": [\'OS\', \'PER\',\'firewall\',\'IP_firewall\',\'routeur\',\'IP_routeur\'],"colors": colors}'

In [14]:
print(f"Result AFTER training:")
doc2 = nlp2(u'Windows 10 est la dernière version du système d\'exploitation Windows.Le pare-feu matériel FortiGate bloque les connexions vers les plages d\'adresses IP 10.0.0.0/8 et 192.168.0.0/16. Dans l\'entreprise XYZ Inc., les serveurs tournent sous Windows Server 2019 pour assurer la stabilité de leur infrastructure réseau. La PME "Techno Innovations" fermera ses portes du 10 au 20 juillet pour participer à une importante conférence internationale sur les nouvelles technologies. Configurez le routeur Linksys avec l\'adresse IP 192.168.0.1 pour accéder à son interface d\'administration. Jean Dupont est un employé. J\'ai été impressionné par les performances améliorées de mon ordinateur depuis que je suis passé à Windows 11. Les imprimantes du sous-réseau 172.16.0.0/16 sont accessibles depuis n\'importe quel appareil du réseau local. Le DNS est responsable de traduire les noms de domaine en adresses IP, ce qui permet aux utilisateurs d\'accéder à des sites web tels que \'www.google.com\' en utilisant le lien \'172.217.167.174\'. Le pare-feu logiciel pfSense est largement utilisé dans les environnements réseau complexes avec la plage d\'adresses IP 172.16.0.0/20. Chez JKLM Hosting, les serveurs tournent sous NixOS 21.05 pour son approche déclarative et son système de gestion de paquets innovant. \"Saveurs d\'Asie\", une PME de restauration asiatique, est ouverte tous les jours de la semaine, de 11h30 à 22h00, pour régaler les amateurs de cuisine asiatique. Le pare-feu logiciel pfSense est largement utilisé dans les environnements réseau complexes avec la plage d\'adresses IP 172.16.0.0/20, et l\'entreprise utilise FreeBSD 13 pour ses serveurs, appréciant la flexibilité et les performances de ce système d\'exploitation open-source.')
print_doc_entities(doc2)

Result AFTER training:
     Windows 10 OS
     Windows OS
     FortiGate Firewall
     10.0.0.0/8 IP_Firewall
     192.168.0.0/16 IP_Firewall
     Windows Server 2019 OS_serveur
     10 au 20 juillet Date
     Linksys Routeur
     192.168.0.1 IP_Routeur
     Jean Dupont Personne
     Windows 11 OS
     172.16.0.0/16 IP_Subnet
     www.google.com DNS
     172.217.167.174 IP_DNS
     pfSense Firewall
     172.16.0.0/20 IP_Firewall
     NixOS 21.05 OS_serveur
     semaine Days
     11h30 Time_Start
     22h00 Time_End
     pfSense Firewall
     172.16.0.0/20 IP_Firewall
     FreeBSD 13 OS_serveur


In [15]:
displacy.serve([doc, doc2], style="ent",auto_select_port=True)

/home/jc/.local/lib/python3.10/site-packages/spacy/displacy/__init__.py:106: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'ent' visualizer
Serving on http://0.0.0.0:5000 ...



127.0.0.1 - - [30/Jul/2023 20:44:50] "GET / HTTP/1.1" 200 17720
127.0.0.1 - - [30/Jul/2023 20:44:50] "GET /favicon.ico HTTP/1.1" 200 17720


Shutting down server on port 5000.


In [16]:
doc3 = nlp(u'Tous les pc de travail de la compta sont sur Windows 10 et tous les postes de travail de l\'usine sont sous Linux. L\'usine et la compta sont sur deux sous-réseaux, 172.16.0.0/16 et 172.17.0.0/16. Les horaires d\'entreprise sont de 8h00 à 21h00. ')
print_doc_entities(doc3)

     Windows 10 OS
     Linux OS
     172.16.0.0/16 IP_Firewall
     172.17.0.0/16 IP_Firewall
     8h00 Time_Start
     21h00 Time_End


In [17]:
doc6 = nlp2(u'Tous les pc de travail de la compta sont sur Windows 10 et tous les postes de travail de l\'usine sont sous Linux. L\'usine et la compta sont sur deux sous-réseaux, 172.16.0.0/16 et 172.17.0.0/16. Les horaires d\'entreprise sont de 8h00 à 21h00. ')
print_doc_entities(doc6)

     Windows 10 OS
     Linux OS
     172.16.0.0/16 IP_Firewall
     172.17.0.0/16 IP_Firewall
     8h00 Time_Start
     21h00 Time_End


In [18]:
doc4 = nlp(u'Au sein de l\'entreprise fictive \"TechNet Corp\", notre réseau informatique joue un rôle essentiel dans le bon fonctionnement de nos opérations quotidiennes. Notre infrastructure de réseau est conçue pour être robuste, sécurisée et hautement performante afin de répondre aux besoins de l\'entreprise en pleine croissance.Nous utilisons des routeurs Cisco de pointe pour assurer une connectivité réseau rapide et fiable à l\'échelle de l\'entreprise. Ces routeurs sont configurés pour optimiser la diffusion du trafic et minimiser les temps d\'attente, garantissant ainsi une expérience utilisateur fluide pour tous nos employés. Les adresses IP des routeurs sont définies dans la plage 192.168.1.0/24.\n\
En ce qui concerne les systèmes d\'exploitation, tous les ordinateurs de nos employés fonctionnent sous Windows 10, car il offre une interface conviviale et une compatibilité élevée avec nos logiciels et applications internes. De plus, nos serveurs sont équipés de la dernière version de Linux pour sa stabilité et ses performances élevées en matière de serveur.\n\
La sécurité de notre réseau est une priorité absolue. Nous avons déployé des pare-feux FortiGate à chaque point d\'accès au réseau pour protéger nos données sensibles contre les menaces potentielles. Ces pare-feux sont configurés pour surveiller en temps réel le trafic réseau et bloquer toute activité suspecte ou malveillante. Les adresses IP des pare-feux sont définies dans les plages 10.0.0.0/8 et 192.168.0.0/16.\n\
John Smith, en tant que responsable du département des ventes, est autorisé à accéder aux données de vente et aux rapports financiers. En revanche, les employés du département marketing n\'ont pas accès à ces informations pour des raisons de confidentialité.\n\
Dans l\'ensemble, notre réseau d\'entreprise fictive \"TechNet Corp\" est conçu pour offrir une connectivité rapide et fiable, une sécurité renforcée et une gestion efficace des ressources. Notre équipe informatique continue de travailler en étroite collaboration pour maintenir notre réseau à la pointe de la technologie et garantir une expérience utilisateur optimale pour tous nos employés.')

In [21]:
doc5 = nlp2(u'Au sein de l\'entreprise fictive \"TechNet Corp\", notre réseau informatique joue un rôle essentiel dans le bon fonctionnement de nos opérations quotidiennes. Notre infrastructure de réseau est conçue pour être robuste, sécurisée et hautement performante afin de répondre aux besoins de l\'entreprise en pleine croissance.Nous utilisons des routeurs Cisco de pointe pour assurer une connectivité réseau rapide et fiable à l\'échelle de l\'entreprise. Ces routeurs sont configurés pour optimiser la diffusion du trafic et minimiser les temps d\'attente, garantissant ainsi une expérience utilisateur fluide pour tous nos employés. Les adresses IP des routeurs sont définies dans la plage 192.168.1.0/24.\n\
En ce qui concerne les systèmes d\'exploitation, tous les ordinateurs de nos employés fonctionnent sous Windows 10, car il offre une interface conviviale et une compatibilité élevée avec nos logiciels et applications internes. De plus, nos serveurs sont équipés de la dernière version de Linux pour sa stabilité et ses performances élevées en matière de serveur.\n\
La sécurité de notre réseau est une priorité absolue. Nous avons déployé des pare-feux FortiGate à chaque point d\'accès au réseau pour protéger nos données sensibles contre les menaces potentielles. Ces pare-feux sont configurés pour surveiller en temps réel le trafic réseau et bloquer toute activité suspecte ou malveillante. Les adresses IP des pare-feux sont définies dans les plages 10.0.0.0/8 et 192.168.0.0/16.\n\
John Smith, en tant que responsable du département des ventes, est autorisé à accéder aux données de vente et aux rapports financiers. En revanche, les employés du département marketing n\'ont pas accès à ces informations pour des raisons de confidentialité.\n\
Dans l\'ensemble, notre réseau d\'entreprise fictive \"TechNet Corp\" est conçu pour offrir une connectivité rapide et fiable, une sécurité renforcée et une gestion efficace des ressources. Notre équipe informatique continue de travailler en étroite collaboration pour maintenir notre réseau à la pointe de la technologie et garantir une expérience utilisateur optimale pour tous nos employés.')

In [22]:
displacy.serve([doc4, doc5], style="ent",auto_select_port=True)


Using the 'ent' visualizer
Serving on http://0.0.0.0:5000 ...



127.0.0.1 - - [30/Jul/2023 20:47:25] "GET / HTTP/1.1" 200 9731
127.0.0.1 - - [30/Jul/2023 20:47:26] "GET /favicon.ico HTTP/1.1" 200 9731


Shutting down server on port 5000.


In [191]:
print_doc_entities(doc4)

     192.168.1.0/24 IP_Routeur
     Windows 10 OS
     Linux OS
     FortiGate Firewall
     10.0.0.0/8 IP_Firewall
     192.168.0.0/16 IP_Firewall
     John Smith Personne
     revanche Days


In [108]:
save_model(nlp,"model-fr-numconseils-v2.zip")

In [208]:
doc7 = nlp(u'Dans le réseau de l\'entreprise fictive \"TechCorp\", un système complexe de gestion est en place pour assurer une connectivité fluide et sécurisée. Les employés utilisent principalement des ordinateurs équipés de Windows 10, MacOS et Linux comme systèmes d\'exploitation. Certains préfèrent également utiliser leurs propres appareils BYOD, tels que des téléphones Android ou des iPhones avec iOS.\n\
Le réseau est divisé en plusieurs sous-réseaux pour des raisons de sécurité et d\'efficacité. Par exemple, le département de la comptabilité se trouve sur le sous-réseau 192.168.1.0/24, tandis que l\'usine est sur le sous-réseau 172.16.1.0/24. Chaque sous-réseau est géré par un routeur dédié, comme le routeur Cisco RV340 pour l\'usine, avec l\'adresse IP 172.16.1.1.\n\
Pour faciliter la gestion des adresses IP, un serveur DHCP est en place. Il attribue automatiquement des adresses IP aux appareils qui se connectent au réseau Wi-Fi. Ainsi, les employés peuvent utiliser leurs appareils BYOD en toute simplicité sans avoir à configurer manuellement une adresse IP. Le serveur DHCP utilise la plage d\'adresses IP 192.168.0.100 à 192.168.0.200 pour les appareils du réseau.\n\
La sécurité est une priorité absolue chez TechCorp. Un pare-feu matériel FortiGate bloque toutes les connexions non autorisées vers le réseau d\'entreprise. Il est configuré pour autoriser uniquement les connexions à partir d\'adresses IP spécifiques. Par exemple, le pare-feu est configuré pour autoriser uniquement les connexions à partir de l\'adresse IP 192.168.9.40.\n\
Les employés peuvent accéder aux serveurs de fichiers et aux autres ressources réseau via des noms de domaine conviviaux tels que "serveur.techcorp.local". Ces noms de domaine sont résolus en adresses IP par un serveur DNS dédié.\n\
Chaque jour, le réseau est surveillé de près par une équipe d\'administrateurs système compétents. Des sauvegardes régulières sont effectuées pour garantir la continuité des opérations en cas de problème.\n\
Dans l\'ensemble, le réseau de TechCorp est bien conçu et géré de manière à fournir une expérience utilisateur fluide et sécurisée tout en répondant aux besoins spécifiques de l\'entreprise.')

In [209]:
doc8 =nlp2(u'Dans le réseau de l\'entreprise fictive \"TechCorp\", un système complexe de gestion est en place pour assurer une connectivité fluide et sécurisée. Les employés utilisent principalement des ordinateurs équipés de Windows 10, MacOS et Linux comme systèmes d\'exploitation. Certains préfèrent également utiliser leurs propres appareils BYOD, tels que des téléphones Android ou des iPhones avec iOS.\n\
Le réseau est divisé en plusieurs sous-réseaux pour des raisons de sécurité et d\'efficacité. Par exemple, le département de la comptabilité se trouve sur le sous-réseau 192.168.1.0/24, tandis que l\'usine est sur le sous-réseau 172.16.1.0/24. Chaque sous-réseau est géré par un routeur dédié, comme le routeur Cisco RV340 pour l\'usine, avec l\'adresse IP 172.16.1.1.\n\
Pour faciliter la gestion des adresses IP, un serveur DHCP est en place. Il attribue automatiquement des adresses IP aux appareils qui se connectent au réseau Wi-Fi. Ainsi, les employés peuvent utiliser leurs appareils BYOD en toute simplicité sans avoir à configurer manuellement une adresse IP. Le serveur DHCP utilise la plage d\'adresses IP 192.168.0.100 à 192.168.0.200 pour les appareils du réseau.\n\
La sécurité est une priorité absolue chez TechCorp. Un pare-feu matériel FortiGate bloque toutes les connexions non autorisées vers le réseau d\'entreprise. Il est configuré pour autoriser uniquement les connexions à partir d\'adresses IP spécifiques. Par exemple, le pare-feu est configuré pour autoriser uniquement les connexions à partir de l\'adresse IP 192.168.9.40.\n\
Les employés peuvent accéder aux serveurs de fichiers et aux autres ressources réseau via des noms de domaine conviviaux tels que "serveur.techcorp.local". Ces noms de domaine sont résolus en adresses IP par un serveur DNS dédié.\n\
Chaque jour, le réseau est surveillé de près par une équipe d\'administrateurs système compétents. Des sauvegardes régulières sont effectuées pour garantir la continuité des opérations en cas de problème.\n\
Dans l\'ensemble, le réseau de TechCorp est bien conçu et géré de manière à fournir une expérience utilisateur fluide et sécurisée tout en répondant aux besoins spécifiques de l\'entreprise.')

## Visualisation web

In [210]:
#html = displacy.render([doc, doc2], style="ent", page=True)

# Sauvegarder le résultat dans un fichier HTML
displacy.serve([doc7, doc8], style="ent")

/home/jc/.local/lib/python3.10/site-packages/spacy/displacy/__init__.py:106: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'ent' visualizer
Serving on http://0.0.0.0:5000 ...



127.0.0.1 - - [27/Jul/2023 11:11:20] "GET / HTTP/1.1" 200 13945
127.0.0.1 - - [27/Jul/2023 11:11:21] "GET /favicon.ico HTTP/1.1" 200 13945


Shutting down server on port 5000.
